<a href="https://colab.research.google.com/github/Navya-1803/NLP-Practice/blob/main/Book_Recommender_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Environment Setup and Data Ingestion

In [ ]:
# Cell 1: Environment Setup and Data Ingestion
import pandas as pd
import numpy as np
import urllib.request
import tarfile
import os

In [ ]:
# 1. Download the standard CMU Book Summary Dataset
url = "http://www.cs.cmu.edu/~dbamman/data/booksummaries.tar.gz"
file_name = "booksummaries.tar.gz"

print("Downloading CMU Book Summaries dataset...")
urllib.request.urlretrieve(url, file_name)

('booksummaries.tar.gz', <http.client.HTTPMessage at 0x7e183975efc0>)

In [ ]:
# 2. Extract the archive
print("Extracting dataset...")
with tarfile.open(file_name, "r:gz") as tar:
    tar.extractall()

Extracting dataset...


/tmp/ipykernel_5562/2641610089.py:4: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()


In [ ]:
# 3. Load the data into a Pandas DataFrame
# The dataset is tab-separated and lacks headers, so we define them manually according to the dataset's documentation.
data_path = "booksummaries/booksummaries.txt"
columns = ['Wikipedia_ID', 'Freebase_ID', 'Title', 'Author', 'Pub_Date', 'Genres', 'Plot']
df = pd.read_csv(data_path, sep='\t', names=columns)

In [ ]:
# 4. Display the initial state to verify ingestion
print(f"\nDataset successfully loaded with {df.shape[0]} books and {df.shape[1]} features.")
display(df[['Title', 'Author', 'Genres', 'Plot']].head(3))


Dataset successfully loaded with 16559 books and 7 features.


,Title,Author,Genres,Plot
0,Animal Farm,George Orwell,"{""/m/016lj8"": ""Roman \u00e0 clef"", ""/m/06nbt"":...","Old Major, the old boar on the Manor Farm, ca..."
1,A Clockwork Orange,Anthony Burgess,"{""/m/06n90"": ""Science Fiction"", ""/m/0l67h"": ""N...","Alex, a teenager living in near-future Englan..."
2,The Plague,Albert Camus,"{""/m/02m4t"": ""Existentialism"", ""/m/02xlf"": ""Fi...",The text of The Plague is divided into five p...


NLP Preprocessing and TF-IDF Vectorization

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# 1. Download required NLTK resources
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

True

In [ ]:
# 2. Define the text cleaning pipeline
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(words)

print(f"Applying NLP cleaning pipeline to all {len(df)} plot summaries. This will take a minute...")
df['Clean_Plot'] = df['Plot'].apply(clean_text)

Applying NLP cleaning pipeline to all 16559 plot summaries. This will take a minute...


In [ ]:
# 3. Initialize and fit the TF-IDF Vectorizer
print("Vectorizing text using TF-IDF...")
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(df['Clean_Plot'])

print(f"TF-IDF Matrix created with shape: {tfidf_matrix.shape}")
print("(Rows = Books, Columns = Unique Words/Bigrams)")

display(df[['Title', 'Plot', 'Clean_Plot']].head(2))

Vectorizing text using TF-IDF...
TF-IDF Matrix created with shape: (16559, 10000)
(Rows = Books, Columns = Unique Words/Bigrams)


,Title,Plot,Clean_Plot
0,Animal Farm,"Old Major, the old boar on the Manor Farm, ca...",old major old boar manor farm call animal farm...
1,A Clockwork Orange,"Alex, a teenager living in near-future Englan...",alex teenager living nearfuture england lead g...


In [ ]:
display(df[['Title', 'Plot', 'Clean_Plot']].head(5))

,Title,Plot,Clean_Plot
0,Animal Farm,"Old Major, the old boar on the Manor Farm, ca...",old major old boar manor farm call animal farm...
1,A Clockwork Orange,"Alex, a teenager living in near-future Englan...",alex teenager living nearfuture england lead g...
2,The Plague,The text of The Plague is divided into five p...,text plague divided five part town oran thousa...
3,An Enquiry Concerning Human Understanding,The argument of the Enquiry proceeds by a ser...,argument enquiry proceeds series incremental s...
4,A Fire Upon the Deep,The novel posits that space around the Milky ...,novel posit space around milky way divided con...


Content-Based Filtering using Cosine Similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [ ]:
print("Calculating full Cosine Similarity matrix (this may take 10-15 seconds)...")
# Calculate the cosine similarity for all ~16,500 books
# To save RAM, we cast the result to float32
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix).astype(np.float32)

print(f"Cosine Similarity Matrix shape: {cosine_sim.shape}")

Calculating full Cosine Similarity matrix (this may take 10-15 seconds)...
Cosine Similarity Matrix shape: (16559, 16559)


In [ ]:
# Create a reverse mapping of book titles to their index
indices = pd.Series(df.index, index=df['Title']).drop_duplicates()

def get_content_recommendations(title, cosine_sim_matrix=cosine_sim, data=df, top_n=5):
    """Returns top N similar books based on semantic plot similarity."""
    if title not in indices:
        return f"Sorry, '{title}' is not in our dataset."

    idx = indices[title]
    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]

    sim_scores = list(enumerate(cosine_sim_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]

    book_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]

    recommendations = data.iloc[book_indices][['Title', 'Author', 'Genres']].copy()
    recommendations['Similarity_Score'] = np.round(scores, 4)

    return recommendations

NameError: name 'pd' is not defined

Interactive Testing with Partial Matching

In [ ]:
import warnings
warnings.filterwarnings('ignore')

def search_book_title(query, data=df):
    """Finds exact book titles in the dataset from a partial string query."""
    matches = data[data['Title'].str.contains(query, case=False, na=False)]
    if matches.empty:
        return None
    return matches['Title'].tolist()

print("==================================================")
print("     📚 NLP CONTENT RECOMMENDER SYSTEM 📚     ")
print("==================================================")

user_query = input("Enter a book title (or part of it) to test: ")
matches = search_book_title(user_query)

if not matches:
    print(f"\n❌ Sorry, no books found containing '{user_query}' in the dataset.")
else:
    target_title = matches[0]

    if len(matches) > 1:
        print(f"\n⚠️ Found {len(matches)} matches (e.g., {', '.join(matches[:3])}).")
        print(f"👉 Defaulting to the first match: '{target_title}'\n")
    else:
        print(f"\n✅ Found match: '{target_title}'\n")

    recommendations = get_content_recommendations(target_title)
    display(recommendations)

     📚 NLP CONTENT RECOMMENDER SYSTEM 📚     
Enter a book title (or part of it) to test: Kite Runner

✅ Found match: 'The Kite Runner'



,Title,Author,Genres,Similarity_Score
12391,The Princes of the Golden Cage,Nathalie Mallet,"{""/m/014dfn"": ""Speculative fiction"", ""/m/01hmn...",0.5385
10806,The Exile Kiss,George Alec Effinger,"{""/m/01qpc"": ""Cyberpunk"", ""/m/06n90"": ""Science...",0.3738
9765,Monkey Bridge,Lan Cao,NaN,0.2503
15220,Mushroom in the Sand,NaN,"{""/m/01jfsb"": ""Thriller"", ""/m/02n4kr"": ""Myster...",0.1672
12193,The Gate House,Nelson DeMille,NaN,0.1418


Synthesizing Ratings & Building Collaborative Filtering (Item-Based)

In [ ]:
import random

print("1. Generating synthetic user ratings for the full dataset...")
all_titles = df['Title'].dropna().unique().tolist()
num_users = 1000  # Increased user count for better matrix density

np.random.seed(42)
random.seed(42)
ratings_data = []

for user_id in range(1, num_users + 1):
    # Each user reads between 15 to 50 random books
    read_books = random.sample(all_titles, random.randint(15, 50))
    for book in read_books:
        ratings_data.append({'User_ID': user_id, 'Title': book, 'Rating': np.random.randint(1, 6)})

df_ratings = pd.DataFrame(ratings_data)

print("2. Creating the User-Item Matrix...")
# To optimize memory during pivot, we ensure the matrix is efficient
user_book_matrix = df_ratings.pivot_table(index='User_ID', columns='Title', values='Rating').fillna(0).astype(np.float32)

print("3. Calculating Item-Item Similarity Matrix...")
# This compares all rated books against each other
book_user_matrix = user_book_matrix.T
cf_cosine_sim = cosine_similarity(book_user_matrix).astype(np.float32)

cf_cosine_sim_df = pd.DataFrame(cf_cosine_sim, index=book_user_matrix.index, columns=book_user_matrix.index)

def get_collaborative_recommendations(title, sim_matrix=cf_cosine_sim_df, top_n=5):
    """Returns top N similar books based on user rating patterns."""
    if title not in sim_matrix.columns:
        return f"Sorry, '{title}' has no synthesized user ratings in our matrix."

    sim_scores = sim_matrix[title].sort_values(ascending=False).drop(title).head(top_n)

    recs = pd.DataFrame(sim_scores).reset_index()
    recs.columns = ['Title', 'CF_Similarity_Score']
    recs['CF_Similarity_Score'] = recs['CF_Similarity_Score'].round(4)
    return recs

print("\nCollaborative Filtering Model Ready!")
print(f"User-Item Matrix shape: {user_book_matrix.shape} (Users x Books)")

# Let's test the interactive title from Cell 4 if it exists, otherwise pick a random one
try:
    print(f"\n--- Top 5 Collaborative Recommendations for: '{target_title}' ---")
    display(get_collaborative_recommendations(target_title))
except NameError:
    test_cf_book = df_ratings['Title'].iloc[0]
    print(f"\n--- Top 5 Collaborative Recommendations for: '{test_cf_book}' ---")
    display(get_collaborative_recommendations(test_cf_book))

1. Generating synthetic user ratings for the full dataset...
2. Creating the User-Item Matrix...
3. Calculating Item-Item Similarity Matrix...

Collaborative Filtering Model Ready!
User-Item Matrix shape: (1000, 14069) (Users x Books)

--- Top 5 Collaborative Recommendations for: 'The Kite Runner' ---


,Title,CF_Similarity_Score
0,Shadow Fires,1.0
1,Les amitiés particulières,1.0
2,The Adventures of Harry Richmond,1.0
3,To Have and To Hold,1.0
4,Only Forward,1.0


Hybrid Recommender System

In [ ]:
def get_hybrid_recommendations(title, content_weight=0.5, cf_weight=0.5, top_n=5):
    """Combines NLP Content-Based scores with Collaborative Filtering scores."""

    print(f"Generating Hybrid Recommendations for '{title}'...")
    print(f"(Weights -> Content: {content_weight*100}%, Collaborative: {cf_weight*100}%)\n")

    # 1. Fetch a MASSIVE pool of recommendations to ensure we don't accidentally drop books
    # We pull 10,000 to basically grab the whole valid dataset
    content_recs = get_content_recommendations(title, top_n=10000)
    cf_recs = get_collaborative_recommendations(title, top_n=10000)

    if isinstance(content_recs, str) or isinstance(cf_recs, str):
        return "Error: Book not found in one or both matrices. Cannot hybridize."

    # 2. Merge using an OUTER join!
    # This ensures if a book is highly relevant in NLP but has a 0 in CF, it survives.
    hybrid_df = pd.merge(content_recs, cf_recs, on='Title', how='outer')

    # 3. Fill missing values
    # If a book was in the NLP list but had no CF rating, give it a CF score of 0 (and vice versa)
    hybrid_df['Similarity_Score'] = hybrid_df['Similarity_Score'].fillna(0)
    hybrid_df['CF_Similarity_Score'] = hybrid_df['CF_Similarity_Score'].fillna(0)

    # Fill missing metadata (like Genres) created by the outer join
    if 'Genres' in hybrid_df.columns:
        hybrid_df['Genres'] = hybrid_df['Genres'].fillna("Unknown/Not Provided")

    # 4. Calculate the weighted Hybrid Score
    hybrid_df['Hybrid_Score'] = (hybrid_df['Similarity_Score'] * content_weight) + (hybrid_df['CF_Similarity_Score'] * cf_weight)

    # 5. Sort by the final Hybrid Score
    hybrid_df = hybrid_df.sort_values(by='Hybrid_Score', ascending=False).head(top_n)

    # Clean up the output
    hybrid_df['Hybrid_Score'] = hybrid_df['Hybrid_Score'].round(4)

    if 'Genres' in hybrid_df.columns:
        final_output = hybrid_df[['Title', 'Genres', 'Similarity_Score', 'CF_Similarity_Score', 'Hybrid_Score']]
    else:
        final_output = hybrid_df[['Title', 'Similarity_Score', 'CF_Similarity_Score', 'Hybrid_Score']]

    return final_output.reset_index(drop=True)

# --- Test the Hybrid Model ---
# 90% Content / 10% CF: This should now heavily favor the Harry Potter sequels!
display(get_hybrid_recommendations(target_title, content_weight=0.7, cf_weight=0.3))

Generating Hybrid Recommendations for 'The Kite Runner'...
(Weights -> Content: 70.0%, Collaborative: 30.0%)



,Title,Genres,Similarity_Score,CF_Similarity_Score,Hybrid_Score
0,The Princes of the Golden Cage,"{""/m/014dfn"": ""Speculative fiction"", ""/m/01hmn...",0.5385,0.0,0.3770
1,Only Forward,"{""/m/06n90"": ""Science Fiction"", ""/m/014dfn"": ""...",0.0176,1.0,0.3123
2,Les amitiés particulières,Unknown/Not Provided,0.0174,1.0,0.3122
3,The Adventures of Harry Richmond,"{""/m/01qxvh"": ""Romance novel"", ""/m/08sdrw"": ""A...",0.0095,1.0,0.3067
4,The Lion's Game,Unknown/Not Provided,0.0000,1.0,0.3000


Model Explainability

In [ ]:
def get_top_keywords(title, tfidf_model=tfidf, tfidf_mat=tfidf_matrix, top_n=10):
    """Extracts the highest-weighted TF-IDF terms for a specific book."""

    if title not in indices:
        return f"Sorry, '{title}' is not in our dataset."

    idx = indices[title]
    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]

    print(f"🔍 Analyzing the algorithm's 'vision' for: '{title}'\n")

    # 1. Get the mathematical vector for this specific book
    book_vector = tfidf_mat[idx].tocoo()

    # 2. Map the vector indices back to the actual English words
    tuples = zip(book_vector.col, book_vector.data)

    # 3. Sort the words by their TF-IDF mathematical weight (highest to lowest)
    sorted_items = sorted(tuples, key=lambda x: (x[1], x[0]), reverse=True)
    feature_names = tfidf_model.get_feature_names_out()

    print(f"Top {top_n} highest-weighted terms (TF-IDF Score):")
    print("-" * 40)

    # 4. Display the results
    for col, score in sorted_items[:top_n]:
        print(f"  • {feature_names[col].ljust(20)} : {score:.4f}")

# --- Test Explainability ---
# Let's look at the target title from our interactive test
try:
    get_top_keywords(target_title)
except NameError:
    get_top_keywords(df['Title'].iloc[0])

🔍 Analyzing the algorithm's 'vision' for: 'Harry Potter and the Philosopher's Stone'

Top 10 highest-weighted terms (TF-IDF Score):
----------------------------------------
  • harry                : 0.7546
  • hermione             : 0.2630
  • snape                : 0.2184
  • dumbledore           : 0.1935
  • voldemort            : 0.1821
  • ron                  : 0.1729
  • stone                : 0.1723
  • professor            : 0.1290
  • harrys               : 0.1080
  • hogwarts             : 0.0967


Advanced Recommender with Metadata Feature Engineering

In [ ]:
import json

def parse_genres(genre_str):
    try:
        # Converts the messy string format into a list of genre names
        genre_dict = json.loads(genre_str.replace("'", '"'))
        return " ".join(genre_dict.values())
    except:
        return ""

print("1. Engineering metadata features (Author + Genres)...")
# Create a specialized 'Metadata' string for every book
df['Clean_Genres'] = df['Genres'].apply(parse_genres)
df['Author_Genre_Soup'] = (df['Author'].fillna("") + " ") + df['Clean_Genres']

# 2. Vectorize the metadata separately
# We use a CountVectorizer here instead of TF-IDF because we want
# an EXACT match on an author's name to be very powerful.
from sklearn.feature_extraction.text import CountVectorizer
count_vec = CountVectorizer(stop_words='english')
metadata_matrix = count_vec.fit_transform(df['Author_Genre_Soup'])

# 3. Calculate Metadata Similarity
metadata_sim = cosine_similarity(metadata_matrix, metadata_matrix).astype(np.float32)

def get_pro_recommendations(title, top_n=5):
    """
    A 'Professional' recommender that uses:
    - 40% NLP Plot Similarity
    - 40% Author/Genre Match
    - 20% Collaborative (Synthetic User) Data
    """
    if title not in indices:
        return f"Book '{title}' not found."

    idx = indices[title]
    if isinstance(idx, pd.Series): idx = idx.iloc[0]

    # Get Scores from all 3 individual engines
    plot_scores = cosine_sim[idx]       # From Cell 3
    meta_scores = metadata_sim[idx]     # From this Cell

    # For Collaborative, we map back to the full dataframe index
    # We'll use a simplified version of our CF logic here
    try:
        cf_row = cf_cosine_sim_df.loc[title]
        cf_scores = np.zeros(len(df))
        for t, score in cf_row.items():
            if t in indices:
                cf_scores[indices[t]] = score
    except:
        cf_scores = np.zeros(len(df))

    # 4. Final Weighted Sum
    # Adjust these weights to change the 'personality' of your recommender
    final_scores = (plot_scores * 0.4) + (meta_scores * 0.4) + (cf_scores * 0.2)

    # 5. Filter and Sort
    sim_scores = list(enumerate(final_scores))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1] # Skip the book itself

    book_indices = [i[0] for i in sim_scores]
    return df.iloc[book_indices][['Title', 'Author', 'Clean_Genres']]

print("Advanced Engine Ready!")
print("\n--- Testing 'Pro' Recommendations (Author + Genre + Plot + User Data) ---")
display(get_pro_recommendations(target_title))

1. Engineering metadata features (Author + Genres)...
Advanced Engine Ready!

--- Testing 'Pro' Recommendations (Author + Genre + Plot + User Data) ---


,Title,Author,Clean_Genres
10806,The Exile Kiss,George Alec Effinger,Cyberpunk Science Fiction Speculative fiction ...
6858,Learning the World,Ken MacLeod,Science Fiction Speculative fiction Fiction Novel
7852,Only Forward,Michael Marshall Smith,Science Fiction Speculative fiction Fantasy
5978,Shadow Fires,Dean Koontz,Horror Fiction Romance novel
9518,A Thousand Splendid Suns,Khaled Hosseini,Fiction Novel
